# Notebook 22 — h2o_dataset_preparation

**Purpose:** Build the labeled dataset for H2O Driverless AI binary classification.  
Target: `tem_sancao` (0 = not sanctioned, 1 = sanctioned).  
Removes leakage columns and ID fields. Exports clean CSV.

**Input:**
- `data/serving/serving_fornecedor_features.parquet`

**Output:**
- `data/serving/h2o_supplier_risk_dataset.csv`

**Columns removed (leakage):**
- `qtd_sancoes`, `tem_ceis`, `tem_cnep`, `sancao_ativa`, `valor_total_multa` — direct leakage from target
- `_loaded_at` — pipeline metadata

**ID Column:**
- `supplier_sk` — included for H2O row identification; mark as "ID Column" in experiment setup

**Predictors (12):**
- Categorical (4): `porte`, `natureza_juridica_desc`, `uf`, `situacao_cadastral`
- Numeric (8): `qtd_contratos`, `qtd_orgaos_distintos`, `valor_total_contratos`,
  `valor_medio_contrato`, `valor_max_contrato`, `anos_ativo_contratos`,
  `tem_contrato_pncp`, `tem_contrato_compras`

**Target (1):** `tem_sancao`

**Notes:**
- Imbalanced dataset: 1.23% positive class — enable imbalanced model in H2O DAI
- Scorer: AUCPR (not AUC) — AUC is misleading for severely imbalanced targets
- No nulls in any column — no imputation needed
- supplier_sk ensures row uniqueness — 469,072 duplicate feature vectors exist
  across 810,921 suppliers (expected: different suppliers with identical profiles)

In [ ]:
## Step 1 — Imports and configuration

import duckdb
from pathlib import Path
import sys

ROOT = None
for candidate in [Path.cwd(), *Path.cwd().parents]:
    if (candidate / "utils" / "paths.py").exists():
        ROOT = candidate
        break
sys.path.insert(0, str(ROOT))

from utils.paths import get_project_root, serving_path, to_sql_path, ensure_dir

ROOT = get_project_root()

INPUT_PATH  = serving_path(ROOT, "serving_fornecedor_features")
OUTPUT_PATH = to_sql_path(ROOT / "data" / "serving" / "h2o_supplier_risk_dataset.csv")

ensure_dir(ROOT / "data" / "serving")

con = duckdb.connect()

print("INPUT :", INPUT_PATH, "| exists:", Path(INPUT_PATH).exists())
print("OUTPUT:", OUTPUT_PATH)

In [ ]:
## Step 2 — Build H2O dataset (remove leakage, keep supplier_sk as ID column)

# supplier_sk incluído como ID column para o H2O
# Não será usado como preditor — configurar como "ID Column" no experiment setup
con.execute(f"""
    COPY (
        SELECT
            -- ID column (marcar como "ID Column" no H2O — não é preditor)
            supplier_sk,

            -- categoricas (4)
            porte,
            natureza_juridica_desc,
            uf,
            situacao_cadastral,

            -- numericas de contrato (8)
            qtd_contratos,
            qtd_orgaos_distintos,
            CAST(valor_total_contratos AS DOUBLE)  AS valor_total_contratos,
            valor_medio_contrato,
            CAST(valor_max_contrato AS DOUBLE)     AS valor_max_contrato,
            anos_ativo_contratos,
            tem_contrato_pncp,
            tem_contrato_compras,

            -- target (1)
            tem_sancao

            -- removidos intencionalmente:
            -- qtd_sancoes       → leakage: deriva do target
            -- tem_ceis          → leakage: componente do target
            -- tem_cnep          → leakage: componente do target
            -- sancao_ativa      → leakage: estado da sanção
            -- valor_total_multa → leakage: existe apenas para sancionados
            -- _loaded_at        → metadado de pipeline

        FROM read_parquet('{INPUT_PATH}')
    )
    TO '{OUTPUT_PATH}'
    (FORMAT CSV, HEADER true, QUOTE '"')
""")

print(f"Written: {OUTPUT_PATH}")
print(f"File size: {Path(OUTPUT_PATH).stat().st_size / 1024 / 1024:.1f} MB")

In [ ]:
## Step 3 — Validation and duplicate check

con.execute(f"""
    CREATE OR REPLACE VIEW h2o_dataset AS
    SELECT * FROM read_csv('{OUTPUT_PATH}',
        header=true, quote='"', delim=','
    )
""")

checks = []

total = con.execute("SELECT COUNT(*) FROM h2o_dataset").fetchone()[0]
checks.append(("row_count = 810921",         total == 810_921,   total))

# 14 colunas: 1 ID + 12 preditoras + 1 target
ncols = len(con.execute("DESCRIBE h2o_dataset").fetchall())
checks.append(("column_count = 14",          ncols == 14,        ncols))

cols = [r[0] for r in con.execute("DESCRIBE h2o_dataset").fetchall()]

# supplier_sk é ID column — presença esperada, não é leakage
leakage = [c for c in ["qtd_sancoes","tem_ceis","tem_cnep",
                        "sancao_ativa","valor_total_multa","_loaded_at"] if c in cols]
checks.append(("no leakage columns",         len(leakage) == 0,  leakage or "clean"))

checks.append(("supplier_sk present (ID)",   "supplier_sk" in cols,  True))
checks.append(("target present",             "tem_sancao" in cols,   True))

pos = con.execute("SELECT COUNT(*) FROM h2o_dataset WHERE tem_sancao = 1").fetchone()[0]
neg = con.execute("SELECT COUNT(*) FROM h2o_dataset WHERE tem_sancao = 0").fetchone()[0]
checks.append(("positives = 9946",           pos == 9_946,       pos))
checks.append(("negatives = 800975",         neg == 800_975,     neg))

null_count = 0
for col in cols:
    null_count += con.execute(f'SELECT COUNT(*) FROM h2o_dataset WHERE "{col}" IS NULL').fetchone()[0]
checks.append(("no nulls",                   null_count == 0,    null_count))

distinct = con.execute("SELECT COUNT(DISTINCT supplier_sk) FROM h2o_dataset").fetchone()[0]
checks.append(("supplier_sk unique",         distinct == total,  distinct))

target_type = con.execute("DESCRIBE h2o_dataset").df().set_index("column_name").loc["tem_sancao","column_type"]
checks.append(("target is numeric",          "INT" in target_type or "BIGINT" in target_type, target_type))

print(f"{'Check':<35} {'Status':<8} {'Value'}")
print("-" * 65)
all_pass = True
for name, passed, value in checks:
    status = "PASS" if passed else "FAIL"
    if not passed:
        all_pass = False
    print(f"{name:<35} {status:<8} {value}")
print("-" * 65)
print("ALL CHECKS PASSED ✅" if all_pass else "⚠️  SOME CHECKS FAILED — review above")

print("\n=== DATASET SUMMARY FOR H2O DRIVERLESS AI ===")
print(f"File     : h2o_supplier_risk_dataset.csv")
print(f"Rows     : {total:,}")
print(f"Columns  : {ncols} (1 ID + 12 predictors + 1 target)")
print(f"ID col   : supplier_sk  → marcar como 'ID Column' no experiment setup")
print(f"Target   : tem_sancao   → 0={neg:,} ({neg/total*100:.1f}%)  1={pos:,} ({pos/total*100:.1f}%)")
print(f"Problem  : Binary Classification — Imbalanced")
print(f"Scorer   : AUCPR")
print(f"\nColumns  : {cols}")

In [ ]:
## Step 4 — Stratified train/test split (80/20)

import pandas as pd
from sklearn.model_selection import train_test_split
from pathlib import Path

TRAIN_PATH = str(OUTPUT_PATH).replace(".csv", "_train.csv")
TEST_PATH  = str(OUTPUT_PATH).replace(".csv", "_test.csv")

df = pd.read_csv(OUTPUT_PATH, quoting=1)  # quoting=1 = QUOTE_ALL

X = df.drop(columns=["tem_sancao"])
y = df["tem_sancao"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=42,
    stratify=y       # preserva proporção 1.2% nos dois splits
)

train_df = pd.concat([X_train, y_train], axis=1)
test_df  = pd.concat([X_test,  y_test],  axis=1)

train_df.to_csv(TRAIN_PATH, index=False)
test_df.to_csv(TEST_PATH,   index=False)

print(f"Train: {len(train_df):,} rows | positives: {y_train.sum():,} ({y_train.mean()*100:.2f}%)")
print(f"Test : {len(test_df):,} rows  | positives: {y_test.sum():,} ({y_test.mean()*100:.2f}%)")
print(f"\nTrain file: {TRAIN_PATH}")
print(f"Test  file: {TEST_PATH}")